# Course 15: Research Engineering Interviews
## Session 11: Behavioral rounds, take-homes and work tests

**Goal:** prepare six stories and complete a 90-minute take-home under its rules.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** the tests and reference solutions here were run once on a CPU (November 2026). Cells marked TODO are stubs for you; try them under a timer before opening the reference solutions at the bottom.

### 1. Six stories (TODO)
For each: situation, task, what **I** did, result and what I learned. Under 250 words.
1. A technical challenge
2. A failure
3. A disagreement
4. A time I helped someone
5. A time I learned something fast
6. The work I am proudest of

### 2. The take-home (90 minutes, alone, no AI tools)
The cell below writes `results.csv`: accuracy for four methods over several seeds. Some rows are missing or duplicated, as real logs are.

**Hand in:** code that reports each method's mean and 95% interval, a plot showing every run, a statement of which differences are supported, and a half-page note on what you did, what you would do with more time, and anything you are unsure of.

In [ ]:
import numpy as np, pandas as pd
rng = np.random.default_rng(7)
rows = []
for method, mu, n in [("baseline", 70.0, 5), ("method_a", 70.4, 5), ("method_b", 72.5, 5), ("method_c", 71.0, 3)]:
    for seed in range(n):
        rows.append({"method": method, "seed": seed, "accuracy": round(rng.normal(mu, 0.8), 2)})
df = pd.DataFrame(rows)
df = pd.concat([df, df.iloc[[2]]], ignore_index=True)        # a duplicated row
df.loc[7, "accuracy"] = np.nan                               # a failed run
df.to_csv("results.csv", index=False)
print(df.head(8))

In [ ]:
# TODO: your solution. Start from a clean read of the file:
# df = pd.read_csv("results.csv")

### Review criteria
Runs from a clean start with one command | handles the edge cases in the data | results stated with uncertainty | note says what was done and not done | code readable | stayed within the rules and time

---
## Reference solutions
Try every problem under the timer first. Then run this section: it defines working versions and runs the same tests.

In [ ]:
from scipy import stats
import matplotlib.pyplot as plt

df = pd.read_csv("results.csv")
n_raw = len(df)
df = df.drop_duplicates(subset=["method", "seed"]).dropna(subset=["accuracy"])
print(f"kept {len(df)} of {n_raw} rows after removing duplicates and failed runs")

def summarize(g):
    n, m = len(g), g.mean()
    half = stats.t.ppf(0.975, n - 1) * g.std(ddof=1) / np.sqrt(n)
    return pd.Series({"n": n, "mean": m, "low": m - half, "high": m + half})

summary = df.groupby("method")["accuracy"].apply(summarize).unstack()
print(summary.round(2))

base = summary.loc["baseline"]
for method, row in summary.iterrows():
    if method != "baseline":
        verdict = "supported" if row["low"] > base["high"] else "not supported at this number of runs"
        print(f"{method} better than baseline: {verdict}")

for i, (method, g) in enumerate(df.groupby("method")):
    plt.scatter([i] * len(g), g["accuracy"], alpha=0.7)
    plt.plot([i - 0.2, i + 0.2], [g["accuracy"].mean()] * 2, color="black")
plt.xticks(range(summary.shape[0]), summary.index); plt.ylabel("accuracy"); plt.savefig("results.png")
print("saved results.png")

Non-overlapping intervals are a strict test. A reference note would also say that one method has fewer runs, so its interval is wide, and that more seeds would be the first thing to add.